# 14 — Are the EPC findings robust to target reliability and record timing?

The main EPC outcome is the mean current energy-efficiency score for all valid certificates linked to each sampled postcode. This notebook checks whether the main conclusions depend on three practical aspects of that outcome: postcodes represented by very few properties, the inclusion of certificates that cannot be linked to a unique property identifier (UPRN), and differences in the typical record year.

Five representative EPC models are refitted with the same held-out borough groups and the same training-fold preprocessing used in the primary Ridge analysis. The purpose is to test the stability of the existing conclusions, not to search for a better model after seeing the results.


## Comparisons included

Four fixed analysis versions are evaluated:

1. **At least three properties:** retain postcodes whose outcome is based on three or more properties.
2. **UPRN-covered main outcome:** retain only postcodes for which a UPRN-only outcome can be calculated, while keeping the original all-certificate outcome.
3. **UPRN-only outcome:** use exactly the same postcodes as comparison 2, but calculate the outcome only from certificates linked to a unique property identifier.
4. **Record-year adjusted:** retain the full EPC sample and add the median certificate year as a control.

The selected models cover compact property controls, DINOv2 and full representation fusion, together with the richer EPC controls and their TESSERA extension. All comparisons retain spatially separated borough validation.


In [ ]:
# Connect Google Drive and load the analysis libraries.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn

from joblib import parallel_backend
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path('/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE')
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({name: getattr(_config, name) for name in dir(_config) if not name.startswith('__')})

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 240)

print('Python:', platform.python_version())
print('numpy:', np.__version__, 'pandas:', pd.__version__, 'sklearn:', sklearn.__version__)

# Use the available CPU and memory rather than leaving the 27 inner fits serial.
# The cap avoids launching more large high-dimensional fits than system RAM can hold.
CPU_COUNT = os.cpu_count() or 1
try:
    RAM_GB = int(Path('/proc/meminfo').read_text().split('MemTotal:')[1].split('kB')[0].strip()) / 1024**2
except Exception:
    RAM_GB = 12.0
if RAM_GB >= 40:
    FAST_N_JOBS = min(8, CPU_COUNT)
elif RAM_GB >= 20:
    FAST_N_JOBS = min(4, CPU_COUNT)
else:
    FAST_N_JOBS = min(2, CPU_COUNT)
FAST_N_JOBS = max(1, FAST_N_JOBS)
print(f'Fast mode: {FAST_N_JOBS} parallel fits | detected CPUs: {CPU_COUNT} | RAM: {RAM_GB:.1f} GB')


## 1. Load the frozen inputs

The canonical model table, feature definitions, borough folds and completed primary Ridge outputs are loaded without alteration. The final Notebook-13 convergence check is also required so that the modelling sequence cannot be treated as complete while a preceding interpretation gate remains open.


In [ ]:
required_paths = [
    FINAL_MODEL_TABLE_PATH,
    EPC_20K_FINAL_CANDIDATE,
    FEATURE_MANIFEST_JSON_PATH,
    RIDGE_OUTER_FOLDS_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
    INCREMENTAL_RESULTS_PATH,
    INCREMENTAL_PREDICTIONS_PATH,
    GATV2_AUDIT_PATH,
]
for path in required_paths:
    assert path.exists(), f'Missing prerequisite: {path}'

source_07_audit = json.loads(INCREMENTAL_AUDIT_PATH.read_text())
source_13_audit = json.loads(GATV2_AUDIT_PATH.read_text())
assert source_07_audit['integrity_gate_pass'] is True
assert source_07_audit['interpretation_gate_pass'] is True
assert source_13_audit['integrity_gate_pass'] is True
assert source_13_audit['interpretation_gate_pass'] is True

df_all = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
df = df_all[df_all['task'].eq('EPC')].copy()
manifest = json.loads(FEATURE_MANIFEST_JSON_PATH.read_text())
source_07_spec = json.loads(INCREMENTAL_RUN_SPEC_PATH.read_text())
source_07_results = pd.read_csv(INCREMENTAL_RESULTS_PATH)
source_07_predictions = pd.read_parquet(INCREMENTAL_PREDICTIONS_PATH)
epc_candidate = pd.read_parquet(EPC_20K_FINAL_CANDIDATE)

target_col = manifest['target_column']
group_col = manifest['group_column']
categorical_master = set(manifest['categorical_columns'])
feature_sets = manifest['feature_sets']

assert len(df) == 20000
assert df['sample_id'].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert {'n_properties', 'median_record_year'}.issubset(df.columns)
assert epc_candidate['sample_id'].is_unique
assert 'uprn_only_target' in epc_candidate.columns

if 'uprn_only_target' not in df.columns:
    add_cols = ['sample_id', 'uprn_only_target']
    if 'uprn_only_n_properties' in epc_candidate.columns:
        add_cols.append('uprn_only_n_properties')
    df = df.merge(epc_candidate[add_cols], on='sample_id', how='left', validate='one_to_one')

df = df.sort_values('sample_id', kind='mergesort').reset_index(drop=True)
assert pd.to_numeric(df['uprn_only_target'], errors='coerce').notna().any()
print('Frozen prerequisite chain: PASS')
print('EPC rows:', len(df))


## 2. Define the five representative models

The compact and richer EPC control families are kept separate. The richer controls include floor area and construction age derived from EPC records, so they are interpreted as a strong within-EPC benchmark rather than as an ordinary deployment baseline.


In [ ]:
SELECTED_MODEL_IDS = [
    'EPC_controls_sparse',
    'EPC_controls_sparse__plus__DINOv2',
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata',
    'EPC_controls_extensive',
    'EPC_controls_extensive__plus__TESSERA',
]

def dedupe(columns):
    return list(dict.fromkeys(columns))

model_features = {
    'EPC_controls_sparse': list(feature_sets['EPC_controls_sparse']),
    'EPC_controls_sparse__plus__DINOv2': dedupe(feature_sets['EPC_controls_sparse'] + feature_sets['DINOv2']),
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata': dedupe(
        feature_sets['EPC_controls_sparse'] + feature_sets['All_representations_plus_SV_metadata']
    ),
    'EPC_controls_extensive': list(feature_sets['EPC_controls_extensive']),
    'EPC_controls_extensive__plus__TESSERA': dedupe(
        feature_sets['EPC_controls_extensive'] + feature_sets['TESSERA']
    ),
}

source_specs = pd.DataFrame(source_07_spec['model_specifications']).set_index('model_id')
assert set(SELECTED_MODEL_IDS).issubset(source_specs.index)

def columns_sha256(columns):
    return hashlib.sha256(json.dumps(list(columns), separators=(',', ':')).encode()).hexdigest()

model_rows = []
for model_id in SELECTED_MODEL_IDS:
    source = source_specs.loc[model_id]
    cols = model_features[model_id]
    assert cols and not [c for c in cols if c not in df.columns]
    assert int(source['n_features_manifest']) == len(cols)
    assert source['feature_columns_sha256'] == columns_sha256(cols)
    model_rows.append({
        'model_id': model_id,
        'baseline_id': source['baseline_id'],
        'control_family': source['control_family'],
        'analysis_role': source['analysis_role'],
        'added_feature_set': source['added_feature_set'],
        'n_features_manifest': len(cols),
        'feature_columns_sha256': columns_sha256(cols),
    })

model_specs = pd.DataFrame(model_rows)
assert len(model_specs) == 5
display(model_specs)


## 3. Construct the four fixed analysis versions

The two UPRN comparisons use exactly the same postcodes and borough-fold assignments. Their difference therefore isolates the outcome definition rather than a change in sample composition. The record-year analysis keeps all rows; missing years, if any, are imputed from the training fold only.


In [ ]:
saved_folds = pd.read_csv(RIDGE_OUTER_FOLDS_PATH)
saved_folds = saved_folds[saved_folds['task'].eq('EPC')][
    ['sample_id', 'outer_fold', 'borough_code', 'target']
].copy()
assert saved_folds['sample_id'].is_unique
assert len(saved_folds) == len(df)

df = df.merge(
    saved_folds[['sample_id', 'outer_fold', 'borough_code']],
    on='sample_id', how='left', validate='one_to_one', suffixes=('', '_frozen')
)
assert df['outer_fold'].notna().all()
assert df['outer_fold'].nunique() == RIDGE_OUTER_SPLITS == 5
frozen_group_col = 'borough_code_frozen' if group_col == 'borough_code' else 'borough_code'
assert df[frozen_group_col].astype(str).eq(df[group_col].astype(str)).all()
assert df.groupby(group_col)['outer_fold'].nunique().eq(1).all()

df['_main_target'] = pd.to_numeric(df[target_col], errors='raise')
df['_uprn_target'] = pd.to_numeric(df['uprn_only_target'], errors='coerce')
df['n_properties'] = pd.to_numeric(df['n_properties'], errors='raise')
df['median_record_year'] = pd.to_numeric(df['median_record_year'], errors='coerce')

uprn_mask = df['_uprn_target'].notna()
protocols = {
    'n3_main_target': {
        'label': 'At least three properties',
        'mask': df['n_properties'].ge(3),
        'target': '_main_target',
        'add_record_year': False,
    },
    'uprn_covered_main_target': {
        'label': 'UPRN-covered rows, original outcome',
        'mask': uprn_mask,
        'target': '_main_target',
        'add_record_year': False,
    },
    'uprn_only_target': {
        'label': 'UPRN-only outcome',
        'mask': uprn_mask,
        'target': '_uprn_target',
        'add_record_year': False,
    },
    'record_year_adjusted': {
        'label': 'Full sample with record-year control',
        'mask': pd.Series(True, index=df.index),
        'target': '_main_target',
        'add_record_year': True,
    },
}

protocol_frames = {}
protocol_rows = []
for protocol_id, spec in protocols.items():
    p = df.loc[spec['mask']].copy().sort_values('sample_id', kind='mergesort').reset_index(drop=True)
    p['_analysis_target'] = pd.to_numeric(p[spec['target']], errors='raise')
    assert p['sample_id'].is_unique and p['_analysis_target'].notna().all()
    assert p['outer_fold'].nunique() == 5
    assert p.groupby(group_col)['outer_fold'].nunique().eq(1).all()
    protocol_frames[protocol_id] = p
    protocol_rows.append({
        'protocol_id': protocol_id,
        'label': spec['label'],
        'n_samples': len(p),
        'n_boroughs': p[group_col].nunique(),
        'target': spec['target'],
        'record_year_added': spec['add_record_year'],
    })

assert protocol_frames['uprn_covered_main_target']['sample_id'].equals(
    protocol_frames['uprn_only_target']['sample_id']
)
protocol_summary = pd.DataFrame(protocol_rows)
display(protocol_summary)


## 4. Preserve missing-data meaning and training-fold preprocessing

Street View absence is represented explicitly rather than treated as an ordinary missing observation. Every learned imputation, scaling and category-encoding step is fitted only on the current outer-training data. Ridge strength is selected with three borough-grouped validation splits inside that training data.


In [ ]:
SV_META_COLS = ['sv_has_streetview', 'sv_n_images', 'sv_min_dist_m', 'sv_mean_dist_m']
SV_CLIP_COLS = feature_sets['StreetView_CLIP_only']

def apply_structural_sv_metadata_fill(frame):
    out = frame.copy()
    has = pd.to_numeric(out['sv_has_streetview'], errors='coerce')
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)
    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all()
    assert clip_all_missing[has.eq(0)].all()
    out['sv_has_streetview'] = has
    for c in SV_META_COLS[1:]:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    no_sv = has.eq(0)
    out.loc[no_sv, 'sv_n_images'] = 0.0
    for c in ['sv_min_dist_m', 'sv_mean_dist_m']:
        out.loc[no_sv & out[c].isna(), c] = float(STREETVIEW_EPC_RADIUS_M)
    return out

def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []
    if numeric_cols:
        transformers.append(('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols))
    if categorical_cols:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
            ('onehot', make_onehot()),
        ]), categorical_cols))
    pre = ColumnTransformer(transformers, remainder='drop', sparse_threshold=0.0)
    return Pipeline([
        ('preprocess', pre),
        ('ridge', Ridge(solver='lsqr', max_iter=5000, tol=1e-4)),
    ])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({'true': True, 'false': False})
    assert mapped.notna().all()
    return mapped.astype(bool)


## 5. Freeze the run identity and expected outputs

Fingerprints record the exact sample membership, alternative outcome values, feature definitions and frozen borough assignments. Existing checkpoints are reused only when all identifiers and held-out sample IDs match this specification.


In [ ]:
analysis_key = pd.concat([
    p[['sample_id', 'outer_fold', group_col, '_analysis_target']].assign(protocol_id=protocol_id)
    for protocol_id, p in protocol_frames.items()
], ignore_index=True).sort_values(['protocol_id', 'sample_id'], kind='mergesort')
analysis_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(analysis_key, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
fold_hash = source_07_spec['outer_fold_assignment_sha256']

model_payload = []
for row in model_specs.itertuples(index=False):
    model_payload.append({
        'model_id': row.model_id,
        'baseline_id': None if pd.isna(row.baseline_id) else row.baseline_id,
        'control_family': row.control_family,
        'analysis_role': row.analysis_role,
        'added_feature_set': None if pd.isna(row.added_feature_set) else row.added_feature_set,
        'n_features_manifest': int(row.n_features_manifest),
        'feature_columns_sha256': row.feature_columns_sha256,
    })

protocol_payload = []
for row in protocol_summary.itertuples(index=False):
    protocol_payload.append({
        'protocol_id': row.protocol_id,
        'label': row.label,
        'n_samples': int(row.n_samples),
        'n_boroughs': int(row.n_boroughs),
        'target': row.target,
        'record_year_added': bool(row.record_year_added),
    })

run_spec = {
    'run_spec_version': '14-v1-2026-08-24',
    'notebook': '14_epc_reliability_and_record_timing_sensitivity.ipynb',
    'source_notebook_07_audit': str(INCREMENTAL_AUDIT_PATH),
    'source_notebook_13_audit': str(GATV2_AUDIT_PATH),
    'analysis_key_sha256': analysis_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_hash,
    'sklearn_version': sklearn.__version__,
    'outer_splits': int(RIDGE_OUTER_SPLITS),
    'inner_splits': int(RIDGE_INNER_SPLITS),
    'alpha_grid': [float(x) for x in RIDGE_ALPHA_GRID],
    'protocols': protocol_payload,
    'model_specifications': model_payload,
    'preprocessing': {
        'numeric_imputation': 'training-fold median',
        'numeric_scaling': 'training-fold StandardScaler',
        'categorical_imputation': 'training-fold constant __MISSING__',
        'categorical_encoding': 'training-fold OneHotEncoder(handle_unknown=ignore)',
        'inner_selection': 'three-fold borough-grouped RMSE',
    },
    'interpretation': {
        'n3_comparison': 'descriptive because sample membership differs from the main run',
        'uprn_target_comparison': 'same samples and folds; paired descriptively by fold',
        'record_year_comparison': 'same samples and folds as Notebook 07; paired descriptively by fold',
        'fold_inference': 'mean, standard deviation and wins; no independent-fold p-values',
    },
}
run_spec_sha256 = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode()).hexdigest()

if EPC_ROBUST_RUN_SPEC_PATH.exists():
    existing = json.loads(EPC_ROBUST_RUN_SPEC_PATH.read_text())
    assert existing == run_spec, 'Existing Notebook-14 checkpoints use a different run specification.'
else:
    atomic_json(run_spec, EPC_ROBUST_RUN_SPEC_PATH)

expected_keys = {
    (protocol_id, model_id, fold)
    for protocol_id in protocols
    for model_id in SELECTED_MODEL_IDS
    for fold in range(RIDGE_OUTER_SPLITS)
}
expected_prediction_rows = sum(len(p) * len(SELECTED_MODEL_IDS) for p in protocol_frames.values())
assert len(expected_keys) == 100
print('Expected outer-fold runs:', len(expected_keys))
print('Expected prediction rows:', expected_prediction_rows)
print('Run-spec SHA256:', run_spec_sha256)


## 6. Fit the resumable nested Ridge models

Each completed held-out fold is written immediately. Re-running the notebook skips a fold only after its prediction file, sample membership and run fingerprint have been validated. The independent alpha and inner-fold fits run in parallel, with the number of workers selected from the available CPU and memory. This changes execution speed only; the folds, preprocessing, alpha grid and fitted models are unchanged.


In [ ]:
if EPC_ROBUST_RESULTS_PATH.exists():
    completed = pd.read_csv(EPC_ROBUST_RESULTS_PATH)
    required = {'protocol_id', 'model_id', 'outer_fold', 'run_spec_sha256'}
    assert required.issubset(completed.columns)
    assert not completed.duplicated(['protocol_id', 'model_id', 'outer_fold']).any()
    assert completed['run_spec_sha256'].eq(run_spec_sha256).all()
    completed['outer_fold'] = completed['outer_fold'].astype(int)
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict('records')

def checkpoint_is_valid(path, protocol_id, model_id, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            'sample_id', 'protocol_id', 'model_id', 'outer_fold', 'borough_code',
            'y_true', 'y_pred', 'run_spec_sha256',
        }
        if not required.issubset(p.columns) or p['sample_id'].duplicated().any():
            return False
        if not p['protocol_id'].eq(protocol_id).all() or not p['model_id'].eq(model_id).all():
            return False
        if not p['outer_fold'].astype(int).eq(outer_fold).all():
            return False
        if not p['run_spec_sha256'].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p['y_true']).all() or not np.isfinite(p['y_pred']).all():
            return False
        return set(p['sample_id'].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

for protocol_id, protocol_spec in protocols.items():
    protocol_df = apply_structural_sv_metadata_fill(protocol_frames[protocol_id])
    y = protocol_df['_analysis_target'].to_numpy()
    groups = protocol_df[group_col].astype(str).to_numpy()
    fold_ids = protocol_df['outer_fold'].astype(int).to_numpy()

    for spec_row in model_specs.itertuples(index=False):
        model_id = spec_row.model_id
        cols = list(model_features[model_id])
        if protocol_spec['add_record_year'] and 'median_record_year' not in cols:
            cols.append('median_record_year')
        X = protocol_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(fold_ids == outer_fold)
            train_idx = np.flatnonzero(fold_ids != outer_fold)
            run_key = (protocol_id, model_id, outer_fold)
            pred_file = EPC_ROBUST_CHUNK_DIR / f'{protocol_id}__{model_id}__fold{outer_fold}.parquet'
            completed_keys_now = {
                (r['protocol_id'], r['model_id'], int(r['outer_fold'])) for r in result_rows
            }
            checkpoint_ok = checkpoint_is_valid(
                pred_file, protocol_id, model_id, outer_fold,
                protocol_df.iloc[test_idx]['sample_id'].to_numpy(),
            )
            if run_key in completed_keys_now and checkpoint_ok:
                print('SKIP validated checkpoint:', run_key)
                continue

            print('\n', protocol_id, '|', model_id, '| outer fold', outer_fold)
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]

            inner = GroupKFold(n_splits=RIDGE_INNER_SPLITS)
            inner_splits = list(inner.split(X_train, y_train, g_train))
            for inner_train, inner_valid in inner_splits:
                assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            pipe = build_pipeline(cols)
            search = GridSearchCV(
                pipe,
                param_grid={'ridge__alpha': RIDGE_ALPHA_GRID},
                scoring='neg_root_mean_squared_error',
                cv=inner_splits,
                refit=True,
                n_jobs=FAST_N_JOBS,
                pre_dispatch=FAST_N_JOBS,
                error_score='raise',
            )
            t0 = time.time()
            # Limit BLAS threads inside each worker to avoid CPU over-subscription.
            with parallel_backend('loky', inner_max_num_threads=1):
                search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx) and np.isfinite(pred).all()

            best_alpha = float(search.best_params_['ridge__alpha'])
            row = {
                'protocol_id': protocol_id,
                'protocol_label': protocol_spec['label'],
                'model_id': model_id,
                'baseline_id': spec_row.baseline_id,
                'control_family': spec_row.control_family,
                'analysis_role': spec_row.analysis_role,
                'added_feature_set': spec_row.added_feature_set,
                'outer_fold': int(outer_fold),
                'n_features': int(len(cols)),
                'n_train': int(len(train_idx)),
                'n_test': int(len(test_idx)),
                'best_alpha': best_alpha,
                'alpha_grid_edge': bool(best_alpha in {float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))}),
                'inner_best_rmse': float(-search.best_score_),
                'r2': float(r2_score(y_test, pred)),
                'rmse': rmse(y_test, pred),
                'mae': float(mean_absolute_error(y_test, pred)),
                'fit_seconds': float(elapsed_s),
                'run_spec_sha256': run_spec_sha256,
            }
            pred_frame = pd.DataFrame({
                'sample_id': protocol_df.iloc[test_idx]['sample_id'].to_numpy(),
                'protocol_id': protocol_id,
                'model_id': model_id,
                'outer_fold': outer_fold,
                'borough_code': groups[test_idx],
                'y_true': y_test,
                'y_pred': pred,
                'run_spec_sha256': run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)
            result_rows = [
                r for r in result_rows
                if (r['protocol_id'], r['model_id'], int(r['outer_fold'])) != run_key
            ]
            result_rows.append(row)
            results_now = pd.DataFrame(result_rows).sort_values(
                ['protocol_id', 'model_id', 'outer_fold'], kind='mergesort'
            )
            atomic_csv(results_now, EPC_ROBUST_RESULTS_PATH)
            print(row)
            del search, pipe, X_train, X_test, pred, pred_frame
            gc.collect()

print('Notebook-14 model fitting is complete or safely checkpointed.')


## 7. Verify every result and held-out prediction

Canonical summaries are created only after all 100 model–fold runs and every expected held-out prediction have been recovered and matched to their exact protocol sample.


In [ ]:
results = pd.read_csv(EPC_ROBUST_RESULTS_PATH)
results['outer_fold'] = results['outer_fold'].astype(int)
assert not results.duplicated(['protocol_id', 'model_id', 'outer_fold']).any()
assert results['run_spec_sha256'].eq(run_spec_sha256).all()
actual_keys = set(map(tuple, results[['protocol_id', 'model_id', 'outer_fold']].to_numpy()))
assert actual_keys == expected_keys, 'Notebook 14 is incomplete: rerun Section 6.'

pred_frames = []
chunk_audit_rows = []
for protocol_id, model_id, outer_fold in sorted(expected_keys):
    path = EPC_ROBUST_CHUNK_DIR / f'{protocol_id}__{model_id}__fold{outer_fold}.parquet'
    assert path.exists(), f'Missing prediction chunk: {path.name}'
    p = pd.read_parquet(path)
    assert p['protocol_id'].eq(protocol_id).all()
    assert p['model_id'].eq(model_id).all()
    assert p['outer_fold'].astype(int).eq(outer_fold).all()
    assert p['run_spec_sha256'].eq(run_spec_sha256).all()
    assert p['sample_id'].is_unique
    assert np.isfinite(p['y_true']).all() and np.isfinite(p['y_pred']).all()

    expected = protocol_frames[protocol_id]
    expected = expected[expected['outer_fold'].astype(int).eq(outer_fold)][
        ['sample_id', group_col, '_analysis_target']
    ].sort_values('sample_id').reset_index(drop=True)
    observed = p[['sample_id', 'borough_code', 'y_true']].sort_values('sample_id').reset_index(drop=True)
    assert expected['sample_id'].equals(observed['sample_id'])
    assert expected[group_col].astype(str).equals(observed['borough_code'].astype(str))
    assert np.allclose(expected['_analysis_target'], observed['y_true'], rtol=0, atol=1e-12)
    chunk_audit_rows.append({'protocol_id': protocol_id, 'model_id': model_id, 'outer_fold': outer_fold, 'n_rows': len(p)})
    pred_frames.append(p)

preds = pd.concat(pred_frames, ignore_index=True)
assert not preds.duplicated(['sample_id', 'protocol_id', 'model_id', 'outer_fold']).any()
assert len(preds) == expected_prediction_rows
print('Validated fold-runs:', len(actual_keys))
print('Validated prediction chunks:', len(chunk_audit_rows))
print('Validated prediction rows:', len(preds))


## 8. Summarise model performance and representation increments

Absolute predictive performance is reported for each analysis version. For the three representation-enhanced specifications, the change from the matching control model is calculated within the same held-out borough fold. Positive changes in R² and negative changes in RMSE or MAE indicate improvement.


In [ ]:
summary_rows = []
for (protocol_id, model_id), g in results.groupby(['protocol_id', 'model_id']):
    pg = preds[(preds['protocol_id'].eq(protocol_id)) & (preds['model_id'].eq(model_id))]
    spec = model_specs[model_specs['model_id'].eq(model_id)].iloc[0]
    summary_rows.append({
        'protocol_id': protocol_id,
        'protocol_label': protocols[protocol_id]['label'],
        'model_id': model_id,
        'baseline_id': spec['baseline_id'],
        'control_family': spec['control_family'],
        'added_feature_set': spec['added_feature_set'],
        'n_samples': len(pg),
        'mean_r2': float(g['r2'].mean()),
        'sd_r2': float(g['r2'].std(ddof=1)),
        'mean_rmse': float(g['rmse'].mean()),
        'sd_rmse': float(g['rmse'].std(ddof=1)),
        'mean_mae': float(g['mae'].mean()),
        'sd_mae': float(g['mae'].std(ddof=1)),
        'pooled_r2': float(r2_score(pg['y_true'], pg['y_pred'])),
        'pooled_rmse': rmse(pg['y_true'], pg['y_pred']),
        'pooled_mae': float(mean_absolute_error(pg['y_true'], pg['y_pred'])),
        'alpha_edge_hits': int(coerce_bool(g['alpha_grid_edge']).sum()),
        'total_fit_minutes': float(g['fit_seconds'].sum() / 60),
    })
summary = pd.DataFrame(summary_rows).sort_values(['protocol_id', 'control_family', 'mean_r2'], ascending=[True, True, False])

increment_rows = []
ENHANCED_MODEL_IDS = [
    'EPC_controls_sparse__plus__DINOv2',
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata',
    'EPC_controls_extensive__plus__TESSERA',
]
# The extensive control model has a sparse-control baseline in Notebook 07
# because it was part of the control ablation. It is a control baseline here,
# not a representation-enhanced specification, so select the three intended
# representation comparisons explicitly.
enhanced_specs = model_specs[model_specs['model_id'].isin(ENHANCED_MODEL_IDS)]
assert len(enhanced_specs) == 3
for protocol_id in protocols:
    for spec in enhanced_specs.itertuples(index=False):
        model_g = results[(results['protocol_id'].eq(protocol_id)) & (results['model_id'].eq(spec.model_id))]
        base_g = results[(results['protocol_id'].eq(protocol_id)) & (results['model_id'].eq(spec.baseline_id))]
        paired = model_g.merge(base_g, on='outer_fold', validate='one_to_one', suffixes=('_model', '_baseline'))
        assert len(paired) == 5
        for row in paired.itertuples(index=False):
            increment_rows.append({
                'protocol_id': protocol_id,
                'model_id': spec.model_id,
                'baseline_id': spec.baseline_id,
                'added_feature_set': spec.added_feature_set,
                'outer_fold': int(row.outer_fold),
                'delta_r2': float(row.r2_model - row.r2_baseline),
                'delta_rmse': float(row.rmse_model - row.rmse_baseline),
                'delta_mae': float(row.mae_model - row.mae_baseline),
            })
incremental = pd.DataFrame(increment_rows)
assert len(incremental) == 60
incremental_summary = incremental.groupby(
    ['protocol_id', 'model_id', 'baseline_id', 'added_feature_set'], as_index=False
).agg(
    mean_delta_r2=('delta_r2', 'mean'),
    sd_delta_r2=('delta_r2', 'std'),
    r2_wins_out_of_5=('delta_r2', lambda x: int((x > 0).sum())),
    mean_delta_rmse=('delta_rmse', 'mean'),
    rmse_wins_out_of_5=('delta_rmse', lambda x: int((x < 0).sum())),
    mean_delta_mae=('delta_mae', 'mean'),
    mae_wins_out_of_5=('delta_mae', lambda x: int((x < 0).sum())),
)

atomic_csv(summary, EPC_ROBUST_SUMMARY_PATH)
atomic_csv(incremental, EPC_ROBUST_INCREMENTAL_FOLD_PATH)
atomic_csv(incremental_summary, EPC_ROBUST_INCREMENTAL_SUMMARY_PATH)
atomic_parquet(preds, EPC_ROBUST_PREDICTIONS_PATH)
display(summary)
display(incremental_summary)


## 9. Isolate outcome-definition and record-year effects

The UPRN comparison holds sample membership fixed and changes only the EPC outcome definition. The record-year comparison holds the original full sample and outcome fixed and adds one time control. Both are summarised fold by fold and interpreted descriptively.


In [ ]:
def paired_protocol_comparison(left_id, right_id, left_label, right_label):
    rows = []
    for model_id in SELECTED_MODEL_IDS:
        left = results[(results['protocol_id'].eq(left_id)) & (results['model_id'].eq(model_id))]
        right = results[(results['protocol_id'].eq(right_id)) & (results['model_id'].eq(model_id))]
        paired = left.merge(right, on='outer_fold', validate='one_to_one', suffixes=('_left', '_right'))
        assert len(paired) == 5
        for row in paired.itertuples(index=False):
            rows.append({
                'model_id': model_id,
                'outer_fold': int(row.outer_fold),
                'left_protocol': left_label,
                'right_protocol': right_label,
                'delta_r2_left_minus_right': float(row.r2_left - row.r2_right),
                'delta_rmse_left_minus_right': float(row.rmse_left - row.rmse_right),
                'delta_mae_left_minus_right': float(row.mae_left - row.mae_right),
            })
    return pd.DataFrame(rows)

target_sensitivity = paired_protocol_comparison(
    'uprn_only_target', 'uprn_covered_main_target',
    'UPRN-only outcome', 'Original outcome on the same rows',
)
assert len(target_sensitivity) == 25

primary_selected = source_07_results[
    source_07_results['task'].eq('EPC') & source_07_results['model_id'].isin(SELECTED_MODEL_IDS)
].copy()
year_new = results[results['protocol_id'].eq('record_year_adjusted')].copy()
year_paired = year_new.merge(
    primary_selected[['model_id', 'outer_fold', 'r2', 'rmse', 'mae']],
    on=['model_id', 'outer_fold'], how='inner', validate='one_to_one', suffixes=('_year_adjusted', '_primary'),
)
assert len(year_paired) == 25
year_sensitivity = year_paired[['model_id', 'outer_fold']].copy()
for metric in ['r2', 'rmse', 'mae']:
    year_sensitivity[f'year_adjusted_{metric}'] = year_paired[f'{metric}_year_adjusted']
    year_sensitivity[f'primary_{metric}'] = year_paired[f'{metric}_primary']
    year_sensitivity[f'delta_{metric}'] = year_paired[f'{metric}_year_adjusted'] - year_paired[f'{metric}_primary']

atomic_csv(target_sensitivity, EPC_TARGET_SENSITIVITY_PATH)
atomic_csv(year_sensitivity, EPC_YEAR_SENSITIVITY_PATH)
display(target_sensitivity.groupby('model_id').agg(
    mean_delta_r2=('delta_r2_left_minus_right', 'mean'),
    mean_delta_rmse=('delta_rmse_left_minus_right', 'mean'),
    mean_delta_mae=('delta_mae_left_minus_right', 'mean'),
))
display(year_sensitivity.groupby('model_id').agg(
    mean_delta_r2=('delta_r2', 'mean'),
    r2_wins_out_of_5=('delta_r2', lambda x: int((x > 0).sum())),
    mean_delta_rmse=('delta_rmse', 'mean'),
    mean_delta_mae=('delta_mae', 'mean'),
))


## 10. Check whether prediction errors vary with record year

The primary held-out predictions from Notebook 07 are linked to each postcode's median certificate year. A rank-based relationship (Spearman correlation) is used because it asks whether errors generally rise or fall with record year without assuming a straight-line relationship. Results are also grouped into four equally sized record-year bands for an interpretable comparison.


In [ ]:
primary_preds = source_07_predictions[
    source_07_predictions['task'].eq('EPC') & source_07_predictions['model_id'].isin(SELECTED_MODEL_IDS)
].copy()
year_lookup = df[['sample_id', 'median_record_year']].copy()
primary_preds = primary_preds.merge(year_lookup, on='sample_id', how='left', validate='many_to_one')
primary_preds['residual'] = primary_preds['y_true'] - primary_preds['y_pred']
primary_preds['absolute_residual'] = primary_preds['residual'].abs()

temporal_rows = []
bin_rows = []
for model_id, g in primary_preds.groupby('model_id'):
    valid = g.dropna(subset=['median_record_year', 'residual', 'absolute_residual']).copy()
    signed = spearmanr(valid['median_record_year'], valid['residual'])
    absolute = spearmanr(valid['median_record_year'], valid['absolute_residual'])
    temporal_rows.append({
        'model_id': model_id,
        'n_with_record_year': len(valid),
        'record_year_coverage_pct': 100 * len(valid) / len(g),
        'rank_relationship_year_signed_error': float(signed.statistic),
        'rank_relationship_year_absolute_error': float(absolute.statistic),
    })
    valid['record_year_band'] = pd.qcut(valid['median_record_year'], q=4, duplicates='drop')
    for band, bg in valid.groupby('record_year_band', observed=True):
        bin_rows.append({
            'model_id': model_id,
            'record_year_band': str(band),
            'n': len(bg),
            'mean_record_year': float(bg['median_record_year'].mean()),
            'mean_signed_error': float(bg['residual'].mean()),
            'mean_absolute_error': float(bg['absolute_residual'].mean()),
            'rmse': rmse(bg['y_true'], bg['y_pred']),
        })

temporal_residual = pd.DataFrame(temporal_rows)
temporal_bins = pd.DataFrame(bin_rows)

matched = df[df['_uprn_target'].notna()].copy()
descriptive = pd.DataFrame([{
    'n_total_epc_postcodes': len(df),
    'n_with_at_least_3_properties': int(df['n_properties'].ge(3).sum()),
    'share_with_at_least_3_properties_pct': float(100 * df['n_properties'].ge(3).mean()),
    'n_with_uprn_only_target': int(df['_uprn_target'].notna().sum()),
    'share_with_uprn_only_target_pct': float(100 * df['_uprn_target'].notna().mean()),
    'mean_uprn_minus_main_target': float((matched['_uprn_target'] - matched['_main_target']).mean()),
    'mean_absolute_uprn_main_difference': float((matched['_uprn_target'] - matched['_main_target']).abs().mean()),
    'record_year_coverage_pct': float(100 * df['median_record_year'].notna().mean()),
    'minimum_median_record_year': float(df['median_record_year'].min()),
    'overall_median_record_year': float(df['median_record_year'].median()),
    'maximum_median_record_year': float(df['median_record_year'].max()),
}])

atomic_csv(temporal_residual, EPC_TEMPORAL_RESIDUAL_PATH)
atomic_csv(temporal_bins, EPC_TEMPORAL_BIN_PATH)
atomic_csv(descriptive, EPC_RELIABILITY_DESCRIPTIVE_PATH)
display(descriptive.T)
display(temporal_residual)
display(temporal_bins)


## 11. Final completeness and interpretation gate

The analysis passes only when all expected runs, prediction rows and planned comparisons are present. If the Ridge strength repeatedly selects an edge of the fixed search range, the numerical outputs remain saved but interpretation pauses until that search range is reviewed.


In [ ]:
edge_counts = (
    results.assign(alpha_grid_edge=coerce_bool(results['alpha_grid_edge']))
    .groupby(['protocol_id', 'model_id'])['alpha_grid_edge']
    .sum().rename('edge_hits').reset_index()
)
repeated_edge = edge_counts[edge_counts['edge_hits'].ge(3)]

audit = {
    'run_spec_path': str(EPC_ROBUST_RUN_SPEC_PATH),
    'run_spec_sha256': run_spec_sha256,
    'analysis_key_sha256': analysis_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_hash,
    'source_07_integrity_gate_pass': bool(source_07_audit['integrity_gate_pass']),
    'source_07_interpretation_gate_pass': bool(source_07_audit['interpretation_gate_pass']),
    'source_13_integrity_gate_pass': bool(source_13_audit['integrity_gate_pass']),
    'source_13_interpretation_gate_pass': bool(source_13_audit['interpretation_gate_pass']),
    'expected_fold_runs': int(len(expected_keys)),
    'completed_fold_runs': int(len(actual_keys)),
    'expected_prediction_rows': int(expected_prediction_rows),
    'actual_prediction_rows': int(len(preds)),
    'prediction_chunks_validated': int(len(chunk_audit_rows)),
    'expected_incremental_fold_rows': 60,
    'actual_incremental_fold_rows': int(len(incremental)),
    'expected_uprn_target_comparison_rows': 25,
    'actual_uprn_target_comparison_rows': int(len(target_sensitivity)),
    'expected_record_year_comparison_rows': 25,
    'actual_record_year_comparison_rows': int(len(year_sensitivity)),
    'temporal_residual_models': int(len(temporal_residual)),
    'n_alpha_grid_edge_hits': int(edge_counts['edge_hits'].sum()),
    'n_protocol_models_with_repeated_edge_hits': int(len(repeated_edge)),
    'integrity_gate_pass': bool(
        len(actual_keys) == len(expected_keys)
        and len(preds) == expected_prediction_rows
        and len(chunk_audit_rows) == len(expected_keys)
        and len(incremental) == 60
        and len(target_sensitivity) == 25
        and len(year_sensitivity) == 25
        and len(temporal_residual) == 5
    ),
    'interpretation_gate_pass': bool(repeated_edge.empty),
}
assert audit['integrity_gate_pass'] is True
atomic_json(audit, EPC_ROBUST_AUDIT_PATH)
display(pd.Series(audit, name='value'))

if not audit['interpretation_gate_pass']:
    display(repeated_edge)
    raise RuntimeError(
        'Integrity PASS, but interpretation is paused because at least one protocol-model '
        'selected an alpha-grid edge in three or more folds.'
    )

print('Notebook 14 integrity gate: PASS')
print('Notebook 14 interpretation gate: PASS')


## 12. Main findings

All 100 planned model–fold fits and 393,635 held-out predictions were completed. The selected 20,000-postcode sample has complete UPRN-only outcome coverage, so the UPRN comparison changes the outcome definition without changing sample membership. Across the five models, switching to the UPRN-only outcome changed mean R² by no more than 0.0022. The EPC findings are therefore insensitive to including the small number of fallback address-linked properties in the postcode outcome.

Restricting the analysis to 18,727 postcodes represented by at least three properties produced higher absolute R², but this is a different and more reliable subset rather than a like-for-like model improvement. The representation increments remained consistent: DINOv2 added 0.233 mean R² to compact controls, full fusion added 0.263, and TESSERA added 0.0066 to the richer controls. Every increment was positive in all five held-out borough folds.

Adding median certificate year to the full-sample models produced only small absolute changes. Mean R² increased by approximately 0.004–0.007 across the five selected specifications, while the representation increments remained close to the primary analysis: 0.205 for DINOv2, 0.238 for full fusion and 0.0062 for TESSERA. Record timing therefore explains a small amount of additional variation but does not change the model ordering or the main representation conclusion.

Median record years span 2012.25 to 2026.41, with a sample median of 2020.72. Across the five models, the rank-based relationship between record year and signed prediction error was weakly positive (0.105–0.147), while its relationship with absolute error was close to zero (-0.050 to -0.002). Older-record groups tended to be over-predicted and newer-record groups under-predicted, but older records were not consistently less accurate. Error magnitude was generally lower in the two middle record-year groups than at the earliest and latest ends. Record timing therefore affects the direction of calibration modestly, rather than producing a simple decline in accuracy with age.

Overall, the principal EPC finding is robust across sample reliability, outcome construction and record timing: learned representations add substantial information beyond compact controls, whereas their incremental value is small once richer EPC-derived property controls are available. These are multi-temporal cross-sectional results and do not identify a causal effect of certificate age.
